In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import torch
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from medmnist import FractureMNIST3D
from MedMNIST3D.utils import Transform3D
import gudhi as gd
from timeit import timeit
from eclayr.cubical import CubECLayr, CubDECC

In [9]:
batch_size = 32
num_iter = 10
interval = [0, 1]
steps = 32

# MNIST

In [34]:
train_data = MNIST(root="./MNIST/dataset/raw/", train=True, download=True, transform=ToTensor())  # shape: (60000, 28, 28)
X = (train_data.data / 255).unsqueeze(1)
dataloader = DataLoader(X, batch_size=batch_size)

## ECC

In [10]:
cubecc = CubECLayr(interval=interval, steps=steps)

def train():
    for batch in dataloader:
        ecc = cubecc(batch)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 2.058864895894658


## DECC

In [11]:
cubecc = CubDECC(interval=interval, steps=steps)

def train():
    for batch in dataloader:
        ecc = cubecc(batch)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 14.015390695794485


## PH

In [35]:
def train():
    for batch in dataloader:
        for data in batch:
            for channel in data:
                cpx = gd.CubicalComplex(vertices=channel)
                ph = cpx.persistence()

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 17.20342168749776


# Synthetic

In [15]:
torch.manual_seed(42)
X = torch.rand(1000, 1, 112, 112)
dataloader = DataLoader(X, batch_size=batch_size)

## ECC

In [16]:
cubecc = CubECLayr(interval=interval, steps=steps)

def train():
    for batch in dataloader:
        ecc = cubecc(batch)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 0.4284263709094375


## DECC

In [17]:
cubecc = CubDECC(interval=interval, steps=steps)

def train():
    for batch in dataloader:
        ecc = cubecc(batch)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 3.996422020799946


## PH

In [18]:
def train():
    for batch in dataloader:
        for data in batch:
            for channel in data:
                cpx = gd.CubicalComplex(vertices=channel)
                ph = cpx.persistence()

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 6.805824245803524


# FractureMNIST3D

In [29]:
train_dataset = FractureMNIST3D(split="train", transform=Transform3D(), download=True, root="./MedMNIST3D/data/", as_rgb=False, size=28)
dataloader = DataLoader(train_dataset, batch_size=batch_size)

Using downloaded and verified file: ./MedMNIST3D/data/fracturemnist3d.npz


## ECC

In [31]:
cubecc = CubECLayr(interval=interval, steps=steps)

def train():
    for X, y in dataloader:
        ecc = cubecc(X)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 1.4240650375024415


## DECC

In [32]:
cubecc = CubDECC(interval=interval, steps=steps)

def train():
    for X, y in dataloader:
        ecc = cubecc(X)

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 12.588111124990974


## PH

In [33]:
def train():
    for X, y in dataloader:
        for data in X:
            for channel in data:
                cpx = gd.CubicalComplex(vertices=channel)
                ph = cpx.persistence()

runtime = timeit(train, number=num_iter)
print("Runtime for each iteration:", runtime / num_iter)

Runtime for each iteration: 28.70580396669684
